In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
from itertools import permutations

In [123]:
df_raw = pd.read_csv('data/combined.csv', sep='\t')

/Users/philippbakendire/Documents/GitHub  Repos/master_medical_decision_support/.venv/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3258: DtypeWarning: Columns (4,19,20,21,22,23,24,25,26) have mixed types.Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [124]:
df

,drug1,object,drug2,precipitant,certainty,contraindication,dateAnnotated,ddiPkEffect,ddiPkMechanism,effectConcept,homepage,label,numericVal,objectUri,pathway,precaution,precipUri,severity,uri,whoAnnotated,source,ddiType,evidence,evidenceSource,evidenceStatement,researchStatementLabel,researchStatement
0,http://bio2rdf.org/drugbank:DB01175,Escitalopram,http://bio2rdf.org/drugbank:DB00338,Omeprazole,None,None,None,None,None,None,None,omeprazole increases the AUC of escitalopram,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations30,None,DIKB,None,None,None,NaN,None,None
1,http://bio2rdf.org/drugbank:DB01175,Escitalopram,http://bio2rdf.org/drugbank:DB01238,Aripiprazole,None,None,None,None,None,None,None,aripiprazole increases the AUC of escitalopram,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations65,None,DIKB,None,None,None,NaN,None,None
2,http://bio2rdf.org/drugbank:DB01175,Escitalopram,http://bio2rdf.org/drugbank:DB00501,Cimetidine,None,None,None,None,None,None,None,cimetidine increases the AUC of escitalopram,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations26,None,DIKB,None,None,None,NaN,None,None
3,http://bio2rdf.org/drugbank:DB01104,Sertraline,http://bio2rdf.org/drugbank:DB00501,Cimetidine,None,None,None,None,None,None,None,cimetidine increases the AUC of sertraline,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations97,None,DIKB,None,None,None,NaN,None,None
4,http://bio2rdf.org/drugbank:DB00404,Alprazolam,http://bio2rdf.org/drugbank:DB00472,Fluoxetine,None,None,None,None,None,None,None,fluoxetine increases the AUC of alprazolam,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations20,None,DIKB,None,None,None,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010173,http://bio2rdf.org/drugbank:DB01012,Cinacalcet,http://bio2rdf.org/drugbank:DB00658,Sevelamer,1.2128e-24,None,None,None,None,asystole,None,None,None,None,None,None,None,NaN,None,None,Twosides,None,None,None,NaN,None,None
1010174,http://bio2rdf.org/drugbank:DB01012,Cinacalcet,http://bio2rdf.org/drugbank:DB00658,Sevelamer,4.94711e-45,None,None,None,None,peritonitis,None,None,None,None,None,None,None,NaN,None,None,Twosides,None,None,None,NaN,None,None
1010175,http://bio2rdf.org/drugbank:DB01012,Cinacalcet,http://bio2rdf.org/drugbank:DB00910,Paricalcitol,3.58071e-16,None,None,None,None,bone marrow failure,None,None,None,None,None,None,None,NaN,None,None,Twosides,None,None,None,NaN,None,None
1010176,http://bio2rdf.org/drugbank:DB00658,Sevelamer,http://bio2rdf.org/drugbank:DB00910,Paricalcitol,9.12183e-28,None,None,None,None,Chronic Kidney Disease,None,None,None,None,None,None,None,NaN,None,None,Twosides,None,None,None,NaN,None,None


In [125]:
# DATA PRE-PROCESSING

df = (
    df_raw.copy()
    # .sample(500)
    
    
    # Get both drug names in a lower-capitalize format since there are cases where the same drug name appears just full caps
    .assign(object=lambda d: d.object.str.lower().str.capitalize())
    .assign(precipitant=lambda d: d.precipitant.str.lower().str.capitalize())
    
    
    # Adjust the severity column and normalize the severity levels
    .assign(severity=lambda d: d.severity.replace(' ', np.nan))
    .assign(severity=lambda d: d.severity.map({
        'Critical': 'High',
        'Significant': 'Medium', 
        '1': 'Low',
        '2': 'Medium',
        '3': 'High',
        'None': np.nan
    }))
    
    .assign(evidenceStatement=lambda d: d.evidenceStatement.replace("None", np.nan))

)

df

,drug1,object,drug2,precipitant,certainty,contraindication,dateAnnotated,ddiPkEffect,ddiPkMechanism,effectConcept,homepage,label,numericVal,objectUri,pathway,precaution,precipUri,severity,uri,whoAnnotated,source,ddiType,evidence,evidenceSource,evidenceStatement,researchStatementLabel,researchStatement
0,http://bio2rdf.org/drugbank:DB01175,Escitalopram,http://bio2rdf.org/drugbank:DB00338,Omeprazole,None,None,None,None,None,None,None,omeprazole increases the AUC of escitalopram,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations30,None,DIKB,None,None,None,NaN,None,None
1,http://bio2rdf.org/drugbank:DB01175,Escitalopram,http://bio2rdf.org/drugbank:DB01238,Aripiprazole,None,None,None,None,None,None,None,aripiprazole increases the AUC of escitalopram,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations65,None,DIKB,None,None,None,NaN,None,None
2,http://bio2rdf.org/drugbank:DB01175,Escitalopram,http://bio2rdf.org/drugbank:DB00501,Cimetidine,None,None,None,None,None,None,None,cimetidine increases the AUC of escitalopram,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations26,None,DIKB,None,None,None,NaN,None,None
3,http://bio2rdf.org/drugbank:DB01104,Sertraline,http://bio2rdf.org/drugbank:DB00501,Cimetidine,None,None,None,None,None,None,None,cimetidine increases the AUC of sertraline,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations97,None,DIKB,None,None,None,NaN,None,None
4,http://bio2rdf.org/drugbank:DB00404,Alprazolam,http://bio2rdf.org/drugbank:DB00472,Fluoxetine,None,None,None,None,None,None,None,fluoxetine increases the AUC of alprazolam,None,None,None,None,None,NaN,http://dbmi-icode-01.dbmi.pitt.edu/dikb/resource/DDIObservations20,None,DIKB,None,None,None,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010173,http://bio2rdf.org/drugbank:DB01012,Cinacalcet,http://bio2rdf.org/drugbank:DB00658,Sevelamer,1.2128e-24,None,None,None,None,asystole,None,None,None,None,None,None,None,NaN,None,None,Twosides,None,None,None,NaN,None,None
1010174,http://bio2rdf.org/drugbank:DB01012,Cinacalcet,http://bio2rdf.org/drugbank:DB00658,Sevelamer,4.94711e-45,None,None,None,None,peritonitis,None,None,None,None,None,None,None,NaN,None,None,Twosides,None,None,None,NaN,None,None
1010175,http://bio2rdf.org/drugbank:DB01012,Cinacalcet,http://bio2rdf.org/drugbank:DB00910,Paricalcitol,3.58071e-16,None,None,None,None,bone marrow failure,None,None,None,None,None,None,None,NaN,None,None,Twosides,None,None,None,NaN,None,None
1010176,http://bio2rdf.org/drugbank:DB00658,Sevelamer,http://bio2rdf.org/drugbank:DB00910,Paricalcitol,9.12183e-28,None,None,None,None,Chronic Kidney Disease,None,None,None,None,None,None,None,NaN,None,None,Twosides,None,None,None,NaN,None,None


In [126]:
df_interesting = df.query('severity.notnull()').copy()

df_interesting.to_csv('data/test.csv')

In [127]:
df.query('severity != "None" and severity.notnull()').groupby('source').severity.value_counts()

source  severity
NDF-RT  Medium      1188
        High         688
OSCAR   Low         5858
        Medium      2799
        High         451
Name: severity, dtype: int64

In [128]:
obj = "Irbesartan"
perc = "Amiloride"

history = [obj,perc, "Atenolol"]
new_drugs = ["Norepinephrine", "Sulindac", "Magnesium_oxide"]

In [129]:

class DDI_Logic():
    
    def __init__(self, 
                 data:pd.DataFrame,
                 patient_history:list
                 ):
        self.data = data
        self.patient_history = patient_history
        self.precipitant = []
        
        
    
    def find_DDI(self, all_drug_permutations:list):
        # find DDIs
        result = []
        
        for object, precipitant in all_drug_permutations:
            df_DDI = self.data.query('object == @object and precipitant == @precipitant')
            
            result_dict = {
                'combination' : [object, precipitant],
                'DDI_found': True if len(df_DDI) == 1 else False,
                'severity': df_DDI.severity.values[0] if len(df_DDI) == 1 else False, 
                'evidenceStatement': df_DDI.evidenceStatement.values[0] if len(df_DDI) == 1 else False
            }
            
            result.append(result_dict)
        return result
    
    def generate_permutations(self, drug_list) -> list:
        return list(permutations(drug_list, 2))
            
        
    def check_DDI_from_history(self) -> None:
        return self.find_DDI(all_drug_permutations=self.generate_permutations(self.patient_history))
        
    
    def check_DDI_from_percipitant(self) -> None:
        return self.find_DDI(all_drug_permutations=self.generate_permutations(self.precipitant))
    
    def check_DDI_with_history(self) -> None:
        # return self.generate_permutations(self.precipitant + self.precipitant)
        return self.find_DDI(all_drug_permutations=self.generate_permutations(self.precipitant + self.precipitant))
    
    def user_input_precipitant(self, new_drug:str)->None:
        self.precipitant.append(new_drug)
        
    def show_drugs(self)->None:
        print(self.precipitant)
        
    
    def run(self):
        pass

    

logic = DDI_Logic(data=df_interesting, patient_history=history)

for d in new_drugs:
    logic.user_input_precipitant(d)

logic.check_DDI_with_history()



    

[{'combination': ['Norepinephrine', 'Sulindac'],
  'DDI_found': False,
  'severity': False,
  'evidenceStatement': False},
 {'combination': ['Norepinephrine', 'Magnesium_oxide'],
  'DDI_found': False,
  'severity': False,
  'evidenceStatement': False},
 {'combination': ['Norepinephrine', 'Norepinephrine'],
  'DDI_found': False,
  'severity': False,
  'evidenceStatement': False},
 {'combination': ['Norepinephrine', 'Sulindac'],
  'DDI_found': False,
  'severity': False,
  'evidenceStatement': False},
 {'combination': ['Norepinephrine', 'Magnesium_oxide'],
  'DDI_found': False,
  'severity': False,
  'evidenceStatement': False},
 {'combination': ['Sulindac', 'Norepinephrine'],
  'DDI_found': False,
  'severity': False,
  'evidenceStatement': False},
 {'combination': ['Sulindac', 'Magnesium_oxide'],
  'DDI_found': False,
  'severity': False,
  'evidenceStatement': False},
 {'combination': ['Sulindac', 'Norepinephrine'],
  'DDI_found': False,
  'severity': False,
  'evidenceStatement': Fal

In [2]:
import pandas as pd
import numpy as np
import itertools
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ==========================================
# 1. READ & PRE-PROCESS THE DATASET
# ==========================================
# Load the raw dataset (Make sure test.csv is in your directory)
df_raw = pd.read_csv('data/test.csv')

# Pre-processing pipeline with severe column mappings fixed
df_interesting = (
    df_raw.copy()
    .assign(object=lambda d: d.object.str.lower().str.capitalize())
    .assign(precipitant=lambda d: d.precipitant.str.lower().str.capitalize())
    .assign(severity=lambda d: d.severity.replace(' ', np.nan))
    .assign(severity=lambda d: d.severity.map({
        'Critical': 'High',
        'Significant': 'Medium', 
        '1': 'Low',
        '2': 'Medium',
        '3': 'High',
        'Low': 'Low',       # Preserves existing 'Low' entries in test.csv
        'Medium': 'Medium', # Preserves existing 'Medium' entries in test.csv
        'High': 'High',     # Preserves existing 'High' entries in test.csv
        'None': np.nan
    }))
    .assign(evidenceStatement=lambda d: d.evidenceStatement.fillna('No statement available.'))
)

# Extract unique drug vocabulary list for our autocomplete search bar
unique_drugs = sorted(list(set(df_interesting['object'].unique()) | set(df_interesting['precipitant'].unique())))


# ==========================================
# 2. THE REFACTORED BACKEND DDI LOGIC CLASS
# ==========================================
class DDI_Logic():
    
    def __init__(self, data: pd.DataFrame, patient_history: list):
        self.data = data
        self.patient_history = [str(d).strip().lower().capitalize() for d in patient_history]
        self.precipitant = [] # New additions
        
    def find_DDI(self, all_drug_permutations: list):
        result = []
        
        for obj, precip in all_drug_permutations:
            df_DDI = self.data.query('object == @obj and precipitant == @precip')
            
            # FIX: Check if empty to prevent calling .values[0] index errors on empty matches
            if not df_DDI.empty:
                row = df_DDI.iloc[0] # Take the first match safely
                result_dict = {
                    'combination': [obj, precip],
                    'DDI_found': True,
                    'severity': row['severity'] if pd.notna(row['severity']) else 'Unknown', 
                    'evidenceStatement': row['evidenceStatement'] if pd.notna(row['evidenceStatement']) else 'No entry statement.'
                }
            else:
                result_dict = {
                    'combination': [obj, precip],
                    'DDI_found': False,
                    'severity': 'None', 
                    'evidenceStatement': 'No interaction found in database.'
                }
            
            result.append(result_dict)
        return result
    
    def generate_permutations(self, drug_list) -> list:
        return list(itertools.permutations(drug_list, 2))
            
    def check_DDI_from_history(self) -> list:
        return self.find_DDI(all_drug_permutations=self.generate_permutations(self.patient_history))
        
    def check_DDI_from_precipitant(self) -> list:
        return self.find_DDI(all_drug_permutations=self.generate_permutations(self.precipitant))
    
    def check_DDI_with_history(self) -> list:
        # FIX: Cross-matches historical drugs with newly added drugs instead of adding precipitants to themselves
        cross_pairs = []
        for h_drug in self.patient_history:
            for p_drug in self.precipitant:
                cross_pairs.append((h_drug, p_drug)) # Does new drug affect history?
                cross_pairs.append((p_drug, h_drug)) # Does history affect new drug?
        return self.find_DDI(all_drug_permutations=cross_pairs)
    
    def user_input_precipitant(self, new_drug: str) -> None:
        clean_drug = str(new_drug).strip().lower().capitalize()
        if clean_drug not in self.precipitant:
            self.precipitant.append(clean_drug)
        
    def show_drugs(self) -> None:
        print("Precipitants currently loaded:", self.precipitant)


# ==========================================
# 3. IPYWIDGETS NOTEBOOK DASHBOARD BACKBONE
# ==========================================

# Initialize our logic class using sample baseline patient history
history_mock = ['Moricizine', 'Ziprasidone']
logic = DDI_Logic(data=df_interesting, patient_history=history_mock)

# UI Elements
title_html = widgets.HTML("<h2 style='color:#4338ca; font-family:sans-serif;'>Jupyter DDI Local Test Environment</h2><hr>")

# Use "Combobox" which is an input bar that has built-in autocomplete/suggestions dropdown
drug_search = widgets.Combobox(
    placeholder='Type to search medicine...',
    options=unique_drugs,
    description='New Med:',
    ensure_option=True,
    disabled=False,
    layout=widgets.Layout(width='400px')
)

add_btn = widgets.Button(description='Add Prescription', button_style='warning', icon='plus')
reset_btn = widgets.Button(description='Reset New Meds', button_style='danger', icon='trash')
input_row = widgets.HBox([drug_search, add_btn, reset_btn])

# Monitors/displays for currently tracked items
status_display = widgets.HTML()
output_display = widgets.Output()

def format_html_table(title, ddi_results):
    """Helper function to cleanly render class outputs inside the cell output window."""
    html = f"<h4 style='font-family:sans-serif; margin-top:15px; color:#1e293b;'>{title}</h4>"
    html += "<table style='width:100%; border-collapse:collapse; font-family:sans-serif; text-align:left; font-size:13px;'>"
    html += "<tr style='background-color:#f1f5f9; border-bottom:2px solid #cbd5e1;'><th style='padding:6px;'>Object (Victim)</th><th style='padding:6px;'>Precipitant (Perpetrator)</th><th style='padding:6px;'>Severity</th><th style='padding:6px;'>Evidence / Clinical Statement</th></tr>"
    
    found_flags = 0
    for res in ddi_results:
        if res['DDI_found']:
            found_flags += 1
            sev = res['severity']
            
            # Apply dynamic text alert styling based on severity level
            color = "#64748b" # Default grey
            if sev == 'High': color = "#dc2626; font-weight:bold;" # Red
            elif sev == 'Medium': color = "#d97706; font-weight:bold;" # Orange
            elif sev == 'Low': color = "#2563eb;" # Blue
                
            html += f"<tr style='border-bottom:1px solid #e2e8f0;'>"
            html += f"<td style='padding:6px; font-weight:600;'>{res['combination'][0]}</td>"
            html += f"<td style='padding:6px; font-weight:600;'>{res['combination'][1]}</td>"
            html += f"<td style='padding:6px; color:{color};'>{sev}</td>"
            html += f"<td style='padding:6px; color:#475569;'>{res['evidenceStatement']}</td>"
            html += "</tr>"
            
    if found_flags == 0:
        html += "<tr><td colspan='4' style='padding:12px; text-align:center; color:#94a3b8; font-style:italic;'>No interactions flagged in this profile subset.</td></tr>"
    
    html += "</table>"
    return html

def update_dashboard():
    # Update current list descriptions
    status_html = f"""
    <div style='font-family:sans-serif; background-color:#f8fafc; padding:10px; border-radius:6px; border:1px solid #e2e8f0; margin-bottom:15px;'>
        <b>Baseline Patient History:</b> {", ".join(logic.patient_history) if logic.patient_history else "None"}<br>
        <b>Newly Added Prescriptions (Precipitants):</b> {", ".join(logic.precipitant) if logic.precipitant else "<span style='color:#94a3b8;'>Empty</span>"}
    </div>
    """
    status_display.value = status_html
    
    # Process outputs through your class methods and render
    with output_display:
        clear_output(wait=True)
        
        # 1. Run check_DDI_with_history()
        res_with_history = logic.check_DDI_with_history()
        display(HTML(format_html_table("⚠️ Results: New Meds vs. Patient History (check_DDI_with_history)", res_with_history)))
        
        # 2. Run check_DDI_from_precipitant()
        res_from_precip = logic.check_DDI_from_precipitant()
        display(HTML(format_html_table("⚡ Results: Internal Clashes Among New Meds (check_DDI_from_precipitant)", res_from_precip)))

# Event handling actions
def on_add_clicked(b):
    drug = drug_search.value
    if drug in unique_drugs:
        logic.user_input_precipitant(drug)
        drug_search.value = '' # Reset input bar
        update_dashboard()

def on_reset_clicked(b):
    logic.precipitant = []
    drug_search.value = ''
    update_dashboard()

add_btn.on_click(on_add_clicked)
reset_btn.on_click(on_reset_clicked)

# Render initial view layout inside the notebook output area
display(title_html, input_row, status_display, output_display)
update_dashboard()

HTML(value="<h2 style='color:#4338ca; font-family:sans-serif;'>Jupyter DDI Local Test Environment</h2><hr>")

HTML(value='')

Output()